# LifeStack Training Notebook
### AI that handles life's worst Fridays
End-to-end training pipeline for the LifeStack simulation engine.

In [ ]:
!pip install groq openai chromadb sentence-transformers gradio matplotlib numpy pydantic openenv -q

In [ ]:
# Upload all LifeStack .py files
from google.colab import files
print('Upload all LifeStack .py files: life_state.py, reward.py, lifestack_env.py, simperson.py, conflict_generator.py, action_space.py, agent.py, memory.py, run_episode.py')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
import os
from google.colab import userdata
# Store your GROQ_API_KEY in Colab Secrets (key icon on left sidebar)
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except:
    os.environ['GROQ_API_KEY'] = 'your_key_here'
    print('⚠️ Add your GROQ_API_KEY to Colab Secrets or paste it above')

In [ ]:
from life_state import LifeMetrics, ResourceBudget, DependencyGraph
from reward import compute_reward
from lifestack_env import LifeStackEnv
from simperson import SimPerson
from conflict_generator import generate_conflict
from action_space import apply_action, EXAMPLE_ACTIONS
from agent import LifeStackAgent
from memory import LifeStackMemory

env = LifeStackEnv()
conflict = generate_conflict(difficulty=3)
person = SimPerson()
print('✅ All modules loaded')
print(f'✅ Test conflict: {conflict.title}')
print(f'✅ Person: {person.get_personality_hint()}')

In [ ]:
from run_episode import run_episode
print('Running 3 sample episodes...\n')
rewards = []
for i, diff in enumerate([2, 3, 5], 1):
    result = run_episode(difficulty=diff, verbose=False)
    rewards.append(result['total_reward'])
    print(f'Episode {i} (difficulty {diff}): reward = {result["total_reward"]:.3f} | person = {result["person"]}')
print(f'\nAverage reward: {sum(rewards)/len(rewards):.3f}')

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

n_episodes = 30
episode_rewards = []
print(f'Training for {n_episodes} episodes...\n')

for ep in range(1, n_episodes + 1):
    difficulty = 1 if ep <= 10 else (2 if ep <= 20 else 3)
    result = run_episode(difficulty=difficulty, verbose=False)
    episode_rewards.append(result['total_reward'])

    if ep % 5 == 0:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(episode_rewards, 'b-', alpha=0.6, label='Episode Reward')
        if len(episode_rewards) >= 5:
            rolling = [sum(episode_rewards[max(0,i-4):i+1])/min(5,i+1) for i in range(len(episode_rewards))]
            ax.plot(rolling, 'r--', label='5-Episode Rolling Avg')
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Total Reward')
        ax.set_title(f'LifeStack Learning Curve (Episode {ep}/{n_episodes})')
        ax.legend()
        plt.tight_layout()
        plt.show()
        print(f'Episode {ep}/{n_episodes} | Reward: {episode_rewards[-1]:.3f} | Avg: {sum(episode_rewards[-5:])/5:.3f}')

In [ ]:
from memory import LifeStackMemory
import shutil, os

print('=== BEFORE vs AFTER MEMORY ===\n')

# Without memory
if os.path.exists('./lifestack_memory'):
    shutil.move('./lifestack_memory', './lifestack_memory_backup')
result_no_mem = run_episode(difficulty=5, verbose=False)
print(f'Without memory | Reward: {result_no_mem["total_reward"]:.3f}')

# With memory
if os.path.exists('./lifestack_memory_backup'):
    shutil.move('./lifestack_memory_backup', './lifestack_memory')
result_with_mem = run_episode(difficulty=5, verbose=False)
print(f'With memory    | Reward: {result_with_mem["total_reward"]:.3f}')

improvement = result_with_mem['total_reward'] - result_no_mem['total_reward']
print(f'Improvement    : {improvement:+.3f}')
print(f'\nMemory stats: {LifeStackMemory().get_stats()}')

# Final Summary
**LifeStack:** Built an AI-driven sandbox for simulating complex life scenarios. It scales based on five fundamental personality traits and models resource budgets.
#### Cited Research:
- Generative Agents (Park et al., 2023)
- Large Language Models as Simulated Economic Agents (Horton, 2023)
- Evaluating LLMs for Social Scenarios (Li et al., 2023)
- Role-Playing in LLMs (Shanahan et al., 2023)

**Reward Improvement:** Evaluated baseline against retrieval-augmented dynamic memories.
**HuggingFace Demo:** Uploaded to HuggingFace Spaces.